# 03 — Model Training & Evaluation
**CFPB Consumer Complaint Classification** (Debt collection vs Credit card)

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## 1. Load cleaned data

In [ ]:
train_df = pd.read_csv('../data/cleaned_train.csv')
test_df = pd.read_csv('../data/cleaned_test.csv')
print('Train:', train_df.shape)
print('Test:', test_df.shape)

## 2. TF-IDF vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

X_train = tfidf.fit_transform(train_df['Consumer complaint narrative'])
X_test = tfidf.transform(test_df['Consumer complaint narrative'])

y_train = train_df['Product']
y_test = test_df['Product']

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)

## 3. Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
print('Model trained.')

## 4. Predict and evaluate on test set

In [ ]:
y_pred = model.predict(X_test)

ml_accuracy = accuracy_score(y_test, y_pred)
ml_precision = precision_score(y_test, y_pred, pos_label='Debt collection')
ml_recall = recall_score(y_test, y_pred, pos_label='Debt collection')
ml_f1 = f1_score(y_test, y_pred, pos_label='Debt collection')

print(f'Accuracy:  {ml_accuracy:.4f}')
print(f'Precision: {ml_precision:.4f}')
print(f'Recall:    {ml_recall:.4f}')
print(f'F1 Score:  {ml_f1:.4f}')
print()
print(classification_report(y_test, y_pred))

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=['Credit card', 'Debt collection'])
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Credit card', 'Debt collection'],
            yticklabels=['Credit card', 'Debt collection'], ax=ax)
ax.set_title('ML Model — Confusion Matrix')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../outputs/ml_confusion_matrix.png', dpi=150)
plt.show()

## 5. Save model, vectorizer, and ML predictions

In [ ]:
joblib.dump(model, '../outputs/logistic_model.joblib')
joblib.dump(tfidf, '../outputs/tfidf_vectorizer.joblib')
print('Model and vectorizer saved to /outputs')

ml_preds_df = pd.DataFrame({
    'narrative': test_df['Consumer complaint narrative'].values,
    'true_label': y_test.values,
    'predicted_label': y_pred
})
ml_preds_df.to_csv('../outputs/ml_predictions.csv', index=False)
print('ML predictions saved to /outputs/ml_predictions.csv')

---
## 6. LLM Classification (Google Gemini) — 300-row sample
**Requires `GEMINI_API_KEY` environment variable to be set.**

In [ ]:
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv('../.env')
api_key = os.environ.get('GEMINI_API_KEY', '')
HAS_API_KEY = bool(api_key)
if not HAS_API_KEY:
    print('WARNING: GEMINI_API_KEY not set. LLM section will be skipped.')
else:
    genai.configure(api_key=api_key)
    llm = genai.GenerativeModel('gemini-2.0-flash')
    print('Gemini API key found. Proceeding with LLM classification.')

In [ ]:
sample_df = test_df.sample(n=300, random_state=42).reset_index(drop=True)
print(f'LLM sample size: {len(sample_df)}')
print(sample_df['Product'].value_counts())

In [ ]:
import time

SYSTEM_PROMPT = (
    "You are a complaint classification system for a financial company. "
    "Classify the following customer complaint into EXACTLY ONE of these two categories: "
    "'Debt collection' or 'Credit card'. Respond with ONLY the category name, nothing else."
)

def classify_with_llm(narrative, retries=3):
    words = str(narrative).split()
    truncated = ' '.join(words[:300])
    for attempt in range(retries):
        try:
            response = llm.generate_content(SYSTEM_PROMPT + "\n\n" + truncated)
            return response.text.strip()
        except Exception as e:
            if 'RESOURCE_EXHAUSTED' in str(e) and attempt < retries - 1:
                wait = 30 * (attempt + 1)
                print(f'  Rate limited, waiting {wait}s...')
                time.sleep(wait)
            else:
                return f'Error: {e}'
    return 'Error: max retries'

if HAS_API_KEY:
    print('Starting LLM classification of 300 rows...')
    llm_predictions = []
    for i, row in sample_df.iterrows():
        pred = classify_with_llm(row['Consumer complaint narrative'])
        llm_predictions.append(pred)
        if (i + 1) % 50 == 0:
            print(f'  {i + 1}/300 done')
        time.sleep(1)

    sample_df['llm_predicted'] = llm_predictions
    print('LLM classification complete.')
else:
    print('Skipped — no API key.')

In [ ]:
if HAS_API_KEY:
    llm_preds_df = pd.DataFrame({
        'narrative': sample_df['Consumer complaint narrative'].values,
        'true_label': sample_df['Product'].values,
        'predicted_label': sample_df['llm_predicted'].values
    })
    llm_preds_df.to_csv('../outputs/llm_predictions.csv', index=False)
    print('LLM predictions saved to /outputs/llm_predictions.csv')
else:
    print('Skipped — no API key.')

## 7. LLM metrics

In [ ]:
if HAS_API_KEY:
    llm_accuracy = accuracy_score(sample_df['Product'], sample_df['llm_predicted'])
    llm_precision = precision_score(sample_df['Product'], sample_df['llm_predicted'], pos_label='Debt collection')
    llm_recall = recall_score(sample_df['Product'], sample_df['llm_predicted'], pos_label='Debt collection')
    llm_f1 = f1_score(sample_df['Product'], sample_df['llm_predicted'], pos_label='Debt collection')

    print(f'LLM Accuracy:  {llm_accuracy:.4f}')
    print(f'LLM Precision: {llm_precision:.4f}')
    print(f'LLM Recall:    {llm_recall:.4f}')
    print(f'LLM F1 Score:  {llm_f1:.4f}')
else:
    print('Skipped — no API key.')

## 8. Side-by-side comparison: ML vs LLM (on same 300 rows)

In [ ]:
X_sample = tfidf.transform(sample_df['Consumer complaint narrative'])
ml_sample_pred = model.predict(X_sample)

ml_sample_acc = accuracy_score(sample_df['Product'], ml_sample_pred)
ml_sample_prec = precision_score(sample_df['Product'], ml_sample_pred, pos_label='Debt collection')
ml_sample_rec = recall_score(sample_df['Product'], ml_sample_pred, pos_label='Debt collection')
ml_sample_f1 = f1_score(sample_df['Product'], ml_sample_pred, pos_label='Debt collection')

if HAS_API_KEY:
    comparison = pd.DataFrame({
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
        'ML (LogReg)': [ml_sample_acc, ml_sample_prec, ml_sample_rec, ml_sample_f1],
        'LLM (Gemini)': [llm_accuracy, llm_precision, llm_recall, llm_f1]
    })
    comparison[['ML (LogReg)', 'LLM (Gemini)']] = comparison[['ML (LogReg)', 'LLM (Gemini)']].round(4)
    print(comparison.to_string(index=False))
else:
    print(f'ML Accuracy on 300-row sample: {ml_sample_acc:.4f}')
    print(f'ML Precision: {ml_sample_prec:.4f}')
    print(f'ML Recall:    {ml_sample_rec:.4f}')
    print(f'ML F1 Score:  {ml_sample_f1:.4f}')
    print('\nLLM comparison skipped — no API key.')

In [ ]:
ml_metrics = {'accuracy': ml_sample_acc, 'precision': ml_sample_prec, 'recall': ml_sample_rec, 'f1': ml_sample_f1}

if HAS_API_KEY:
    llm_metrics = {'accuracy': llm_accuracy, 'precision': llm_precision, 'recall': llm_recall, 'f1': llm_f1}
else:
    llm_metrics = {'accuracy': 0, 'precision': 0, 'recall': 0, 'f1': 0}

metrics = {'ml': ml_metrics, 'llm': llm_metrics}
with open('../outputs/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to /outputs/metrics.json')

## 9. Disagreements — 5 examples where ML and LLM disagree

In [ ]:
if HAS_API_KEY:
    sample_df['ml_predicted'] = ml_sample_pred
    disagreements = sample_df[sample_df['ml_predicted'] != sample_df['llm_predicted']]
    print(f'Total disagreements: {len(disagreements)} / 300')
    print()

    for idx, row in disagreements.head(5).iterrows():
        print(f'--- Row {idx} ---')
        print(f'True:     {row["Product"]}')
        print(f'ML pred:  {row["ml_predicted"]}')
        print(f'LLM pred: {row["llm_predicted"]}')
        print(f'Narrative: {row["Consumer complaint narrative"][:200]}...')
        print()
else:
    print('Skipped — no API key. Set GEMINI_API_KEY and re-run from cell 6.')